# Data Deduplication for Machine Learning

Data deduplication is the process of identifying and removing duplicate records from a dataset. In real-world data science, duplicates often arise from system glitches, repeated form submissions, multiple sensor transmissions, or data merging from disparate sources.

---

## Learning Objectives
By the end of this notebook, you will be able to:
1. Distinguish between **full-row duplicates** and **key-based (subset) duplicates**.
2. Identify duplicates using pandas `duplicated()` with different `keep` strategies (`'first'`, `'last'`, `False`).
3. Safely remove duplicates using `drop_duplicates()` and maintain clean DataFrame indices.
4. Understand why deduplication is **critical before train/test splitting** to prevent **data leakage** and evaluation bias.

---
## 1. Why Deduplication Matters in Machine Learning

Failing to remove duplicate records before machine learning modeling causes three serious problems:

1. **Data Leakage & Inflated Evaluation**: If a duplicated record appears in both the training set and the test set, the model is evaluated on data it has already memorized. This yields unrealistically high test scores that fail in production.
2. **Model Bias / Overfitting**: Over-representing identical samples gives them disproportionate weight during model optimization, biasing predictions toward those duplicates.
3. **Wasted Compute & Storage**: Redundant records increase memory footprint and prolong model training times with zero added information.

> **Golden Rule**: Always perform deduplication on the raw data **before** splitting into train and test sets.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Set display options for clean output
pd.set_option('display.max_columns', None)

---
## 2. Creating a Realistic Customer Dataset

Let's construct a sample customer subscription dataset that illustrates both:
- **Exact full-row duplicates**: The entire row is repeated identically.
- **Partial/subset duplicates**: Rows share a unique business key (e.g., `Email` or `CustomerID`), but differ in other fields (e.g., plan updates or multiple signups).

In [2]:
# Sample customer records
raw_data = {
    'CustomerID': [101, 102, 103, 104, 101, 105, 106, 104],
    'Name': ['Ahmed Ali', 'Fatima Hassan', 'Omar Ibrahim', 'Mona Zaki', 'Ahmed Ali', 'Fatima Hassan', 'Tariq Mansoor', 'Mona Zaki'],
    'Email': ['ahmed@example.com', 'fatima@example.com', 'omar@example.com', 'mona@example.com', 
              'ahmed@example.com', 'fatima.h@work.com', 'tariq@example.com', 'mona@example.com'],
    'Plan': ['Basic', 'Premium', 'Basic', 'Standard', 'Basic', 'Standard', 'Premium', 'Standard'],
    'MonthlySpend': [29.99, 79.99, 29.99, 49.99, 29.99, 49.99, 89.99, 49.99],
    'Churn': [0, 0, 1, 0, 0, 1, 0, 0]
}

df = pd.DataFrame(raw_data)
print(f"Total rows in raw dataset: {len(df)}")
df

Total rows in raw dataset: 8


,CustomerID,Name,Email,Plan,MonthlySpend,Churn
0,101,Ahmed Ali,ahmed@example.com,Basic,29.99,0
1,102,Fatima Hassan,fatima@example.com,Premium,79.99,0
2,103,Omar Ibrahim,omar@example.com,Basic,29.99,1
3,104,Mona Zaki,mona@example.com,Standard,49.99,0
4,101,Ahmed Ali,ahmed@example.com,Basic,29.99,0
5,105,Fatima Hassan,fatima.h@work.com,Standard,49.99,1
6,106,Tariq Mansoor,tariq@example.com,Premium,89.99,0
7,104,Mona Zaki,mona@example.com,Standard,49.99,0


---
## 3. Identifying Duplicates with `df.duplicated()`

Pandas provides `df.duplicated()` which returns a boolean Series indicating whether each row is a duplicate.

The behavior is controlled by the `keep` parameter:
- `keep='first'` (default): Marks duplicates as `True` **except for the first occurrence**.
- `keep='last'`: Marks duplicates as `True` **except for the last occurrence**.
- `keep=False`: Marks **all duplicate occurrences as `True`** (best practice for inspecting and auditing conflicting records).

In [3]:
# Check total count of full-row duplicates (keep='first')
num_full_dups = df.duplicated().sum()
print(f"Number of exact full-row duplicate rows: {num_full_dups}")

# Inspect all instances of full-row duplicates using keep=False
print("\nAll instances of full-row duplicates:")
df[df.duplicated(keep=False)]

Number of exact full-row duplicate rows: 2

All instances of full-row duplicates:


,CustomerID,Name,Email,Plan,MonthlySpend,Churn
0,101,Ahmed Ali,ahmed@example.com,Basic,29.99,0
3,104,Mona Zaki,mona@example.com,Standard,49.99,0
4,101,Ahmed Ali,ahmed@example.com,Basic,29.99,0
7,104,Mona Zaki,mona@example.com,Standard,49.99,0


### Identifying Duplicates on Specific Columns (`subset`)

In real applications, an updated record might have a new `Plan` or different `Email`, but the same primary identifier (`CustomerID`). We use the `subset` parameter to catch key-based duplicates.

In [4]:
# Check duplicates based on 'CustomerID'
dup_ids = df.duplicated(subset=['CustomerID'], keep=False)
print(f"Rows with duplicated CustomerID: {dup_ids.sum()}")
df[dup_ids]

Rows with duplicated CustomerID: 4


,CustomerID,Name,Email,Plan,MonthlySpend,Churn
0,101,Ahmed Ali,ahmed@example.com,Basic,29.99,0
3,104,Mona Zaki,mona@example.com,Standard,49.99,0
4,101,Ahmed Ali,ahmed@example.com,Basic,29.99,0
7,104,Mona Zaki,mona@example.com,Standard,49.99,0


In [5]:
# Check duplicates based on 'Email'
dup_emails = df.duplicated(subset=['Email'], keep=False)
print(f"Rows with duplicated Email: {dup_emails.sum()}")
df[dup_emails]

Rows with duplicated Email: 4


,CustomerID,Name,Email,Plan,MonthlySpend,Churn
0,101,Ahmed Ali,ahmed@example.com,Basic,29.99,0
3,104,Mona Zaki,mona@example.com,Standard,49.99,0
4,101,Ahmed Ali,ahmed@example.com,Basic,29.99,0
7,104,Mona Zaki,mona@example.com,Standard,49.99,0


---
## 4. Removing Duplicates with `df.drop_duplicates()`

Once duplicates have been audited, we remove them using `df.drop_duplicates()`.

Important parameters:
- `subset`: List of columns to evaluate for uniqueness. If omitted, all columns must match.
- `keep`: Which occurrence to retain (`'first'`, `'last'`, or `False` to drop all copies).
- `ignore_index=True`: Resets index automatically from `0` to `n-1` (avoids missing or discontinuous index labels).

In [6]:
# 1. Drop exact full-row duplicates
df_no_exact = df.drop_duplicates(ignore_index=True)
print(f"Shape after dropping exact duplicates: {df_no_exact.shape}")
df_no_exact

Shape after dropping exact duplicates: (6, 6)


,CustomerID,Name,Email,Plan,MonthlySpend,Churn
0,101,Ahmed Ali,ahmed@example.com,Basic,29.99,0
1,102,Fatima Hassan,fatima@example.com,Premium,79.99,0
2,103,Omar Ibrahim,omar@example.com,Basic,29.99,1
3,104,Mona Zaki,mona@example.com,Standard,49.99,0
4,105,Fatima Hassan,fatima.h@work.com,Standard,49.99,1
5,106,Tariq Mansoor,tariq@example.com,Premium,89.99,0


In [7]:
# 2. Drop duplicates based on unique identifier (CustomerID), keeping the first occurrence
df_cleaned = df.drop_duplicates(subset=['CustomerID'], keep='first', ignore_index=True)
print(f"Shape after deduplicating by CustomerID: {df_cleaned.shape}")
df_cleaned

Shape after deduplicating by CustomerID: (6, 6)


,CustomerID,Name,Email,Plan,MonthlySpend,Churn
0,101,Ahmed Ali,ahmed@example.com,Basic,29.99,0
1,102,Fatima Hassan,fatima@example.com,Premium,79.99,0
2,103,Omar Ibrahim,omar@example.com,Basic,29.99,1
3,104,Mona Zaki,mona@example.com,Standard,49.99,0
4,105,Fatima Hassan,fatima.h@work.com,Standard,49.99,1
5,106,Tariq Mansoor,tariq@example.com,Premium,89.99,0


---
## 5. Machine Learning Impact: Demonstrating Data Leakage Prevention

Let's examine what happens when we split data **with** vs **without** prior deduplication.

### Scenario A (Flawed Workflow): Train/Test Split on Duplicate Data

In [8]:
# Simulating a larger dataset with duplicate entries
np.random.seed(42)
customer_pool = pd.DataFrame({
    'CustomerID': [101, 102, 103, 104, 105, 106, 107, 108],
    'Feature1': [12.5, 45.2, 33.1, 88.0, 15.3, 62.4, 71.1, 24.8],
    'Feature2': [1.1, 3.4, 2.2, 5.8, 1.3, 4.1, 4.9, 1.9],
    'Target': [0, 1, 0, 1, 0, 1, 1, 0]
})

# Artificially duplicate rows 0 and 3
leaky_df = pd.concat([customer_pool, customer_pool.iloc[[0, 3]]], ignore_index=True)
print(f"Dataset with duplicates: {len(leaky_df)} rows")

# Train/test split WITHOUT deduplication
train_leaky, test_leaky = train_test_split(leaky_df, test_size=0.3, random_state=42)

# Check for overlap between train and test sets
train_ids = set(train_leaky['CustomerID'])
test_ids = set(test_leaky['CustomerID'])
overlap = train_ids.intersection(test_ids)

print(f"Overlapping CustomerIDs in both Train and Test sets: {overlap}")
print("-> WARNING: Data Leakage detected! The test set contains customers already in the training set.")

Dataset with duplicates: 10 rows
Overlapping CustomerIDs in both Train and Test sets: {101}
-> WARNING: Data Leakage detected! The test set contains customers already in the training set.


### Scenario B (Best Practice Workflow): Deduplicate BEFORE Splitting

In [9]:
# Step 1: Deduplicate raw dataset first
clean_df = leaky_df.drop_duplicates(subset=['CustomerID'], keep='first', ignore_index=True)
print(f"Deduplicated dataset: {len(clean_df)} unique rows")

# Step 2: Split the deduplicated data
train_clean, test_clean = train_test_split(clean_df, test_size=0.3, random_state=42)

# Verify zero overlap
train_clean_ids = set(train_clean['CustomerID'])
test_clean_ids = set(test_clean['CustomerID'])
clean_overlap = train_clean_ids.intersection(test_clean_ids)

print(f"Overlapping CustomerIDs: {clean_overlap}")
print("-> SUCCESS: Zero data leakage. Test set represents genuinely unseen records.")

Deduplicated dataset: 8 unique rows
Overlapping CustomerIDs: set()
-> SUCCESS: Zero data leakage. Test set represents genuinely unseen records.


---
## 6. Summary & Best Practices Checklist

| Action | Recommended Method | Note |
| :--- | :--- | :--- |
| **Audit all duplicate records** | `df[df.duplicated(keep=False)]` | Displays all copies for human review before dropping. |
| **Audit key duplicates** | `df[df.duplicated(subset=['key'], keep=False)]` | Finds conflicting records sharing an ID/email. |
| **Drop exact duplicates** | `df.drop_duplicates(ignore_index=True)` | Preserves clean integer indexing. |
| **Drop key duplicates** | `df.drop_duplicates(subset=['key'], keep='first', ignore_index=True)` | Retains the first (or latest if sorted by date) record. |
| **ML Workflow Order** | **Deduplicate first $\to$ Split train/test $\to$ Preprocess** | Strictly prevents evaluation leakage and bias. |

### When to keep duplicates:
- In event/transaction logs where repeated identical events are valid business transactions (e.g. repeated purchases of the same item). In such cases, aggregate or group the transactions rather than dropping them.
